In [4]:
import re
import csv

# مسیرها
input_path = r"F:\Thesis\project\2-RAG\raw_laws\Constitutional law\qanun-asasi.txt"
output_tsv = r"F:\Thesis\project\2-RAG\raw_laws\Constitutional law\preprocessed_data\qanun-asasi_articles.tsv"

LAW_CODE = "qanun_asasi"
LAW_NAME = "قانون اساسی جمهوری اسلامی ایران"


def normalize_digits(text: str) -> str:
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    return text.translate(str.maketrans(persian_digits, english_digits))


def normalize_persian(text: str) -> str:
    text = text.replace("ي", "ی").replace("ك", "ک")
    # حذف کاراکترهای کنترلی جهت‌دهنده
    text = text.replace("\u200F", "")  # Right-to-Left Mark
    text = text.replace("\u200E", "")  # Left-to-Right Mark (اختیاری)
    text = text.replace("\u200C", "")  # Zero Width Non-Joiner (اختیاری)
    text = re.sub(r"\s+", " ", text)
    return text.strip()



# 1) خواندن
with open(input_path, "r", encoding="utf-8") as f:
    raw = f.read()

raw = normalize_digits(raw)
raw = raw.replace("–", "-").replace("—", "-").replace("ـ", "-")

# 2) شکستن خطوط برای متادیتاها
patterns_to_break = [
    # بخش‌های متنی قبل از فصل‌ها (مقدمه، طلیعه نهضت، و...)
    r"((?:مقدمه|طلیعه نهضت|حکومت اسلامی|خشم ملت|بهایی که ملت پرداخت|شیوه حکومت در اسلام|ولایت فقیه|اقتصاد وسیله است نه هدف|زن در قانون اساسی|ارتش مکتبی|قضا در قانون اساسی|قوه مجریه|وسایل ارتباط جمعی|نمایندگان)[^\n]*)",
    # فصل
    r"(فصل\s+(?:اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|یازدهم|دوازدهم|سیزدهم|چهاردهم)\s*[:‏-][^\n]*)",
    # مبحث
    r"(مبحث\s+(?:اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم)\s*[-:‏][^\n]*)",
    # اصل (با اعداد فارسی کامل)
    r"(اصل‏?\s+(?:اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|یازدهم|دوازدهم|سیزدهم|چهاردهم|پانزدهم|شانزدهم|هفدهم|هجدهم|نوزدهم|بیستم|بیست و یکم|بیست و دوم|بیست و سوم|بیست و چهارم|بیست و پنجم|بیست و ششم|بیست و هفتم|بیست و هشتم|بیست و نهم|سی‌ام|سی و یکم|سی و دوم|سی و سوم|سی و چهارم|سی و پنجم|سی و ششم|سی و هفتم|سی و هشتم|سی و نهم|چهلم|چهل و یکم|چهل و دوم|چهل و سوم|چهل و چهارم|چهل و پنجم|چهل و ششم|چهل و هفتم|چهل و هشتم|چهل و نهم|پنجاهم|پنجاه و یکم|پنجاه و دوم|پنجاه و سوم|پنجاه و چهارم|پنجاه و پنجم|پنجاه و ششم|پنجاه و هفتم|پنجاه و هشتم|پنجاه و نهم|شصتم|شصت و یکم|شصت و دوم|شصت و سوم|شصت و چهارم|شصت و پنجم|شصت و ششم|شصت و هفتم|شصت و هشتم|شصت و نهم|هفتادم|هفتاد و یکم|هفتاد و دوم|هفتاد و سوم|هفتاد و چهارم|هفتاد و پنجم|هفتاد و ششم|هفتاد و هفتم|هفتاد و هشتم|هفتاد و نهم|هشتادم|هشتاد و یکم|هشتاد و دوم|هشتاد و سوم|هشتاد و چهارم|هشتاد و پنجم|هشتاد و ششم|هشتاد و هفتم|هشتاد و هشتم|هشتاد و نهم|نودم|نود و یکم|نود و دوم|نود و سوم|نود و چهارم|نود و پنجم|نود و ششم|نود و هفتم|نود و هشتم|نود و نهم|یکصدم|یکصد و یکم|یکصد و دوم|یکصد و سوم|یکصد و چهارم|یکصد و پنجم|یکصد و ششم|یکصد و هفتم|یکصد و هشتم|یکصد و نهم|یکصد و دهم|یکصد و یازدهم|یکصد و دوازدهم|یکصد و سیزدهم|یکصد و چهاردهم|یکصد و پانزدهم|یکصد و شانزدهم|یکصد و هفدهم|یکصد و هجدهم|یکصد و نوزدهم|یکصد و بیستم|یکصد و بیست و یکم|یکصد و بیست و دوم|یکصد و بیست و سوم|یکصد و بیست و چهارم|یکصد و بیست و پنجم|یکصد و بیست و ششم|یکصد و بیست و هفتم|یکصد و بیست و هشتم|یکصد و بیست و نهم|یکصد و سی‌ام|یکصد و سی و یکم|یکصد و سی و دوم|یکصد و سی و سوم|یکصد و سی و چهارم|یکصد و سی و پنجم|یکصد و سی و ششم|یکصد و سی و هفتم|یکصد و سی و هشتم|یکصد و سی و نهم|یکصد و چهلم|یکصد و چهل و یکم|یکصد و چهل و دوم|یکصد و چهل و سوم|یکصد و چهل و چهارم|یکصد و چهل و پنجم|یکصد و چهل و ششم|یکصد و چهل و هفتم|یکصد و چهل و هشتم|یکصد و چهل و نهم|یکصد و پنجاهم|یکصد و پنجاه و یکم|یکصد و پنجاه و دوم|یکصد و پنجاه و سوم|یکصد و پنجاه و چهارم|یکصد و پنجاه و پنجم|یکصد و پنجاه و ششم|یکصد و پنجاه و هفتم|یکصد و پنجاه و هشتم|یکصد و پنجاه و نهم|یکصد و شصتم|یکصد و شصت و یکم|یکصد و شصت و دوم|یکصد و شصت و سوم|یکصد و شصت و چهارم|یکصد و شصت و پنجم|یکصد و شصت و ششم|یکصد و شصت و هفتم|یکصد و شصت و هشتم|یکصد و شصت و نهم|یکصد و هفتادم|یکصد و هفتاد و یکم|یکصد و هفتاد و دوم|یکصد و هفتاد و سوم|یکصد و هفتاد و چهارم|یکصد و هفتاد و پنجم|یکصد و هفتاد و ششم|یکصد و هفتاد و هفتم)[^\n]*)"
]

for p in patterns_to_break:
    raw = re.sub(r"(?<!^)" + p, r"\n\1", raw, flags=re.MULTILINE | re.UNICODE)

lines = raw.splitlines()

# 3) الگوهای دقیق
# بخش‌های متنی
text_section_pattern = re.compile(
    r"^(مقدمه|طلیعه نهضت|حکومت اسلامی|خشم ملت|بهایی که ملت پرداخت|شیوه حکومت در اسلام|ولایت فقیه|اقتصاد وسیله است نه هدف|زن در قانون اساسی|ارتش مکتبی|قضا در قانون اساسی|قوه مجریه|وسایل ارتباط جمعی|نمایندگان)$"
)

# فصل
chapter_pattern = re.compile(
    r"^فصل\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم|یازدهم|دوازدهم|سیزدهم|چهاردهم)\s*[:‏-]\s*(.*)$"
)

# مبحث
section_pattern = re.compile(
    r"^مبحث\s+(اول|دوم|سوم|چهارم|پنجم|ششم|هفتم|هشتم|نهم|دهم)\s*[-:‏]\s*(.*)$"
)

# اصل
principle_pattern = re.compile(
    r"^اصل‏?\s+(.+?)(?:\s*[-:].*)?$"
)

# متغیرهای سلسله‌مراتب
current_text_section = ""  # بخش متنی (مقدمه، طلیعه، و...)
current_chapter = ""        # فصل
current_section = ""        # مبحث
current_principle_num = None
current_principle_text = ""

articles = []

# دیکشنری تبدیل اعداد فارسی به انگلیسی
persian_to_number = {
    "اول": 1, "دوم": 2, "سوم": 3, "چهارم": 4, "پنجم": 5, "ششم": 6, "هفتم": 7, "هشتم": 8, "نهم": 9, "دهم": 10,
    "یازدهم": 11, "دوازدهم": 12, "سیزدهم": 13, "چهاردهم": 14, "پانزدهم": 15, "شانزدهم": 16, "هفدهم": 17, "هجدهم": 18, "نوزدهم": 19, "بیستم": 20,
    "بیست و یکم": 21, "بیست و دوم": 22, "بیست و سوم": 23, "بیست و چهارم": 24, "بیست و پنجم": 25, "بیست و ششم": 26, "بیست و هفتم": 27, "بیست و هشتم": 28, "بیست و نهم": 29,
    "سی‌ام": 30, "سی و یکم": 31, "سی و دوم": 32, "سی و سوم": 33, "سی و چهارم": 34, "سی و پنجم": 35, "سی و ششم": 36, "سی و هفتم": 37, "سی و هشتم": 38, "سی و نهم": 39,
    "چهلم": 40, "چهل و یکم": 41, "چهل و دوم": 42, "چهل و سوم": 43, "چهل و چهارم": 44, "چهل و پنجم": 45, "چهل و ششم": 46, "چهل و هفتم": 47, "چهل و هشتم": 48, "چهل و نهم": 49,
    "پنجاهم": 50, "پنجاه و یکم": 51, "پنجاه و دوم": 52, "پنجاه و سوم": 53, "پنجاه و چهارم": 54, "پنجاه و پنجم": 55, "پنجاه و ششم": 56, "پنجاه و هفتم": 57, "پنجاه و هشتم": 58, "پنجاه و نهم": 59,
    "شصتم": 60, "شصت و یکم": 61, "شصت و دوم": 62, "شصت و سوم": 63, "شصت و چهارم": 64, "شصت و پنجم": 65, "شصت و ششم": 66, "شصت و هفتم": 67, "شصت و هشتم": 68, "شصت و نهم": 69,
    "هفتادم": 70, "هفتاد و یکم": 71, "هفتاد و دوم": 72, "هفتاد و سوم": 73, "هفتاد و چهارم": 74, "هفتاد و پنجم": 75, "هفتاد و ششم": 76, "هفتاد و هفتم": 77, "هفتاد و هشتم": 78, "هفتاد و نهم": 79,
    "هشتادم": 80, "هشتاد و یکم": 81, "هشتاد و دوم": 82, "هشتاد و سوم": 83, "هشتاد و چهارم": 84, "هشتاد و پنجم": 85, "هشتاد و ششم": 86, "هشتاد و هفتم": 87, "هشتاد و هشتم": 88, "هشتاد و نهم": 89,
    "نودم": 90, "نود و یکم": 91, "نود و دوم": 92, "نود و سوم": 93, "نود و چهارم": 94, "نود و پنجم": 95, "نود و ششم": 96, "نود و هفتم": 97, "نود و هشتم": 98, "نود و نهم": 99,
    "یکصدم": 100, "یکصد و یکم": 101, "یکصد و دوم": 102, "یکصد و سوم": 103, "یکصد و چهارم": 104, "یکصد و پنجم": 105, "یکصد و ششم": 106, "یکصد و هفتم": 107, "یکصد و هشتم": 108, "یکصد و نهم": 109, "یکصد و دهم": 110,
    "یکصد و یازدهم": 111, "یکصد و دوازدهم": 112, "یکصد و سیزدهم": 113, "یکصد و چهاردهم": 114, "یکصد و پانزدهم": 115, "یکصد و شانزدهم": 116, "یکصد و هفدهم": 117, "یکصد و هجدهم": 118, "یکصد و نوزدهم": 119, "یکصد و بیستم": 120,
    "یکصد و بیست و یکم": 121, "یکصد و بیست و دوم": 122, "یکصد و بیست و سوم": 123, "یکصد و بیست و چهارم": 124, "یکصد و بیست و پنجم": 125, "یکصد و بیست و ششم": 126, "یکصد و بیست و هفتم": 127, "یکصد و بیست و هشتم": 128, "یکصد و بیست و نهم": 129,
    "یکصد و سی‌ام": 130, "یکصد و سی و یکم": 131, "یکصد و سی و دوم": 132, "یکصد و سی و سوم": 133, "یکصد و سی و چهارم": 134, "یکصد و سی و پنجم": 135, "یکصد و سی و ششم": 136, "یکصد و سی و هفتم": 137, "یکصد و سی و هشتم": 138, "یکصد و سی و نهم": 139,
    "یکصد و چهلم": 140, "یکصد و چهل و یکم": 141, "یکصد و چهل و دوم": 142, "یکصد و چهل و سوم": 143, "یکصد و چهل و چهارم": 144, "یکصد و چهل و پنجم": 145, "یکصد و چهل و ششم": 146, "یکصد و چهل و هفتم": 147, "یکصد و چهل و هشتم": 148, "یکصد و چهل و نهم": 149,
    "یکصد و پنجاهم": 150, "یکصد و پنجاه و یکم": 151, "یکصد و پنجاه و دوم": 152, "یکصد و پنجاه و سوم": 153, "یکصد و پنجاه و چهارم": 154, "یکصد و پنجاه و پنجم": 155, "یکصد و پنجاه و ششم": 156, "یکصد و پنجاه و هفتم": 157, "یکصد و پنجاه و هشتم": 158, "یکصد و پنجاه و نهم": 159,
    "یکصد و شصتم": 160, "یکصد و شصت و یکم": 161, "یکصد و شصت و دوم": 162, "یکصد و شصت و سوم": 163, "یکصد و شصت و چهارم": 164, "یکصد و شصت و پنجم": 165, "یکصد و شصت و ششم": 166, "یکصد و شصت و هفتم": 167, "یکصد و شصت و هشتم": 168, "یکصد و شصت و نهم": 169,
    "یکصد و هفتادم": 170, "یکصد و هفتاد و یکم": 171, "یکصد و هفتاد و دوم": 172, "یکصد و هفتاد و سوم": 173, "یکصد و هفتاد و چهارم": 174, "یکصد و هفتاد و پنجم": 175, "یکصد و هفتاد و ششم": 176, "یکصد و هفتاد و هفتم": 177
}


def flush_principle():
    global current_principle_num, current_principle_text
    if current_principle_num is not None and current_principle_text.strip():
        articles.append({
            "law_code": LAW_CODE,
            "law_name": LAW_NAME,
            "text_section": current_text_section,
            "chapter": current_chapter,
            "section": current_section,
            "principle_number": current_principle_num,
            "text": normalize_persian(current_principle_text),
        })
    current_principle_num = None
    current_principle_text = ""


# پردازش
for line in lines:
    stripped = line.strip()
    if not stripped:
        continue

    # 1) بخش متنی (مقدمه، طلیعه، و...)
    m = text_section_pattern.match(stripped)
    if m:
        flush_principle()
        current_text_section = stripped
        current_chapter = ""
        current_section = ""
        continue

    # 2) فصل
    m = chapter_pattern.match(stripped)
    if m:
        flush_principle()
        current_chapter = stripped
        current_text_section = ""
        current_section = ""
        continue

    # 3) مبحث
    m = section_pattern.match(stripped)
    if m:
        flush_principle()
        current_section = stripped
        continue

    # 4) اصل
    m = principle_pattern.match(stripped)
    if m:
        flush_principle()
        persian_num = m.group(1).strip()
        
        # تبدیل عدد فارسی به انگلیسی
        if persian_num in persian_to_number:
            current_principle_num = persian_to_number[persian_num]
        else:
            current_principle_num = None
        
        current_principle_text = f"اصل {persian_num}"
        continue

    # 5) ادامه متن اصل یا بخش متنی
    if current_principle_num is not None:
        current_principle_text += " " + stripped
    elif current_text_section:
        # ذخیره متن‌های مقدمه و سایر بخش‌های متنی
        if not articles or articles[-1]["text_section"] != current_text_section:
            articles.append({
                "law_code": LAW_CODE,
                "law_name": LAW_NAME,
                "text_section": current_text_section,
                "chapter": "",
                "section": "",
                "principle_number": None,
                "text": stripped,
            })
        else:
            articles[-1]["text"] += " " + stripped

# آخرین اصل
flush_principle()

# خروجی TSV
fieldnames = ["law_code", "law_name", "text_section", "chapter", "section", "principle_number", "text"]

with open(output_tsv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    for row in articles:
        writer.writerow(row)

print("✓ پردازش قانون اساسی جمهوری اسلامی ایران تمام شد.")
print(f"✓ تعداد اصول و بخش‌های متنی استخراج شده: {len(articles)}")
print(f"✓ خروجی TSV: {output_tsv}")


✓ پردازش قانون اساسی جمهوری اسلامی ایران تمام شد.
✓ تعداد اصول و بخش‌های متنی استخراج شده: 178
✓ خروجی TSV: F:\Thesis\project\2-RAG\raw_laws\Constitutional law\preprocessed_data\qanun-asasi_articles.tsv
